## Padding

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3)

x = torch.randn(1, 3, 32, 32)  # 每批次一张图片，3 个通道，大小是 32x32
y = conv(x)

print(y.shape) # 一张图片经过卷积层后，变成了 16 张特征图，每张特征图的大小是 30x30
plt.imshow(y[0, 0].detach().numpy(), cmap="gray") # 显示第一张特征图
plt.title("Feature Map 1")
plt.axis("off")

In [ ]:
import torch
import matplotlib.pyplot as plt

# 1. 生成假图：(批次, 通道, 高, 宽) = (1, 3, 32, 32)
x = torch.randn(1, 3, 32, 32)

# 2. 把张量转成可以画的格式
# 先把批次去掉，变成 (3, 32, 32)
img_tensor = x[0]

# 把 PyTorch 的 [C, H, W] 转成 Matplotlib 需要的 [H, W, C]
img_np = img_tensor.permute(1, 2, 0).numpy()

# 3. 把随机值缩到 0-1 之间，不然颜色会很奇怪
img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min())

# 4. 画出来
plt.figure(figsize=(4, 4))
plt.imshow(img_np)
plt.axis('off')  # 关掉坐标轴
plt.title("Random Fake Image")
plt.show()

## 卷积核

In [ ]:
# 导入 PyTorch 核心库，用来搭建神经网络
import torch

# 导入神经网络层模块，这里用来定义卷积层
import torch.nn as nn

# 导入画图工具库，用来显示图片和卷积提取的特征图
import matplotlib.pyplot as plt

# 导入 torchvision，专门用来处理图像、加载数据集
import torchvision

# 导入图像预处理工具，比如把图片转成张量
import torchvision.transforms as transforms

# ===================== 1. 数据准备 =====================
# 定义图像预处理操作：只做一件事 -> 把图片转成 PyTorch 张量（Tensor）
# 张量就是模型能看懂的数字格式
transform = transforms.Compose([
    transforms.ToTensor()
])

# 加载 CIFAR-10 测试数据集（1万张图片）
# root='./data'：数据集存放在当前目录的 data 文件夹下
# train=False：加载**测试集**（试卷），不是训练集（课本）
# download=True：如果本地没有数据集，自动下载；有就直接加载
# transform=transform：加载图片时，自动执行上面定义的预处理（转张量）
dataset = torchvision.datasets.CIFAR10(
    root='.././data',
    train=False,
    download=True,
    transform=transform
)

# 从测试集中取出**第 0 张**图片和它对应的标签（类别）
img, label = dataset[1]

# 给图片增加一个 batch 维度，把形状从 [3, 32, 32] 变成 [1, 3, 32, 32]
# 因为神经网络模型默认接受一批数据输入，单张图片也要加个维度
x = img.unsqueeze(0)

# ===================== 2. 搭建卷积层 =====================
# 创建一个卷积层 Conv2d
# 3：输入通道数（彩色图片 R、G、B 3个通道）
# 8：输出通道数（卷积后生成 8 张特征图）
# kernel_size=3：卷积核大小 3x3
# padding=1：给图片边缘填充 1 圈像素，保证卷积后图片大小不变
conv = nn.Conv2d(3, 8, kernel_size=3, padding=1)

# ===================== 3. 卷积运算（前向传播） =====================
# 把图片 x 输入到卷积层，得到输出 y
# y 就是卷积提取后的**特征图**，形状是 [1, 8, 32, 32]
y = conv(x)

# ===================== 4. 显示原始图片 =====================
# img.permute(1, 2, 0)：把张量形状从 [3, 32, 32] 转成 [32, 32, 3]
# 因为画图工具要求 宽×高×通道 格式
plt.imshow(img.permute(1, 2, 0))

# 设置图片标题
plt.title("Original Image")

# 关闭坐标轴，让图片更好看
plt.axis('off')

# 弹出窗口显示原图
plt.show()

# ===================== 5. 显示卷积提取的特征图 =====================
# 创建一个 1行6列 的子图窗口，用来显示 6 张特征图
fig, axes = plt.subplots(1, 8, figsize=(12, 4))

# 循环显示前 8 张特征图
for i in range(8):
    # y[0, i]：取第 0 个batch 的第 i 张特征图
    # detach().numpy()：把张量转成普通数字数组，才能画图
    # cmap='gray'：用灰度图显示（黑白）
    axes[i].imshow(y[0, i].detach().numpy(), cmap='gray')
    
    # 关闭子图坐标轴
    axes[i].axis('off')

# 弹出窗口显示所有特征图
plt.show()

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
from PIL import Image
import requests
from io import BytesIO

# 👉 换成国内超快、绝对能打开的猫咪图片！
url = "https://picsum.photos/id/40/500/300"

# 读取图片
response = requests.get(url)
img = Image.open(BytesIO(response.content)).convert("RGB")
print(f"Original image size: {img.size}")
plt.figure()
plt.imshow(img)
plt.show()   # 出图1

# 预处理
transform = transforms.Compose([
    transforms.Resize((32, 32)),  # 改成和 CIFAR 一样大小
    transforms.ToTensor(),
])

img_tensor: torch.Tensor = transform(img) # type: ignore
x = img_tensor.unsqueeze(0)
print(f"Preprocessed image shape: {x.shape}")  # 出图2

# 卷积层
conv = nn.Conv2d(3, 8, kernel_size=3, padding=1)

# 卷积运算
y = conv(x)

# 显示原图
plt.imshow(img_tensor.permute(1, 2, 0))
plt.title("Original Image")
plt.axis("off")
plt.show()

# 显示卷积特征
fig, axes = plt.subplots(1, 8, figsize=(12, 4))
for i in range(8):
    axes[i].imshow(y[0, i].detach().numpy(), cmap="gray")
    axes[i].axis("off")
plt.show()

# Loss 和 梯度

In [ ]:
import torch
from torch import Tensor


def train_simple_model() -> None:
    # =========================================
    # 1. 初始化参数
    # =========================================
    w: Tensor = torch.tensor([0.0], requires_grad=True) 

    # =========================================
    # 2. 构造数据
    # =========================================
    x: Tensor = torch.tensor([1.0])
    y_true: Tensor = torch.tensor([2.0])

    # =========================================
    # 3. 训练参数
    # =========================================
    learning_rate: float = 0.1
    num_steps: int = 100

    # =========================================
    # 4. 训练循环
    # =========================================
    for step in range(num_steps):
        # ---------- 前向传播 ----------
        y_pred: Tensor = w * x

        # ---------- 计算损失 ----------
        loss: Tensor = (y_pred - y_true) ** 2 # 均方误差损失函数，衡量预测值和真实值的差距，值越小表示模型预测越准确

        # ---------- 反向传播 ----------
        loss.backward()

        # ---------- 打印 ----------
        print(f"step {step}")
        print(f"  w        = {w.item():.4f}")
        print(f"  y_pred   = {y_pred.item():.4f}")
        print(f"  loss     = {loss.item():.4f}")
        print(f"  gradient = {w.grad.item():.4f}")  # type: ignore
        print("-" * 30)

        # ---------- 更新参数 ----------
        with torch.no_grad():
            w -= learning_rate * w.grad  # type: ignore

        # ---------- 清梯度 ----------
        w.grad.zero_()  # type: ignore


if __name__ == "__main__":
    train_simple_model()

很好，这一步我就**逐句带你走一遍**，不跳步骤，不讲抽象概念，只讲：

👉 **这一行在干嘛 + 为什么要有它**

我们用你刚才那段最小代码👇

---

# 🧪 原代码（我们逐行拆）

```python
import torch
from torch import Tensor


def train_simple_model() -> None:
```

---

## 🧩 1️⃣ 导入库

```python
import torch
from torch import Tensor
```

👉 意思：

* `torch`：PyTorch库
* `Tensor`：就是“张量”（你可以理解成数组/数字容器）

---

## 🧩 2️⃣ 定义函数

```python
def train_simple_model() -> None:
```

👉 意思：

* 定义一个函数
* `-> None`：这个函数不返回东西

👉 你可以暂时当成：

👉 “把代码包起来，方便执行”

---

# 🧩 3️⃣ 初始化参数

```python
w: Tensor = torch.tensor([0.0], requires_grad=True)
```

👉 这行非常重要，我们慢慢拆：

---

### 🔍 `torch.tensor([0.0])`

👉 创建一个数：

```text
w = 0
```

---

### 🔍 `requires_grad=True`

👉 意思：

👉
**“我之后要对这个变量求梯度（让它学习）”**

👉 如果不写这一句：

❌ 模型就学不了

---

### 🧠 总结这一行

👉
**创建一个可以被训练的参数 w，初始值是 0**

---

# 🧩 4️⃣ 准备数据

```python
x: Tensor = torch.tensor([1.0])
y_true: Tensor = torch.tensor([2.0])
```

👉 意思：

我们造一个数据：

```text
x = 1
正确答案 y = 2
```

👉 也就是：

👉 **真实关系是：y = 2x**

---

# 🧩 5️⃣ 设置训练参数

```python
learning_rate: float = 0.1
num_steps: int = 10
```

👉 意思：

* `learning_rate`：每次改多少（步子大小）
* `num_steps`：训练多少次

---

# 🧩 6️⃣ 开始训练循环

```python
for step in range(num_steps):
```

👉 意思：

👉 **重复做 10 次“试错 + 调整”**

---

# 🔥 从这里开始是核心（每一步训练）

---

## 🧩 7️⃣ 前向传播（预测）

```python
y_pred: Tensor = w * x
```

👉 意思：

👉 用当前的 w 来预测：

```text
预测 = w × x
```

---

👉 第一次时：

```text
w = 0
x = 1
→ y_pred = 0
```

👉 明显错了（应该是 2）

---

# 🧩 8️⃣ 计算损失

```python
loss: Tensor = (y_pred - y_true) ** 2
```

👉 意思：

👉 算“你错多少”

---

👉 第一次：

```text
y_pred = 0
y_true = 2

loss = (0 - 2)^2 = 4
```

👉 错得很大

---

# 🧩 9️⃣ 反向传播（核心！）

```python
loss.backward()
```

👉 这一句做的事：

👉
**自动计算：如果我改 w，会对 loss 有什么影响**

---

👉 它会算出：

```text
w.grad = -4
```

---

👉 这个 -4 的意思是：

👉
**“你应该把 w 往变大的方向改”**

---

# 🧩 🔟 打印（帮助你理解）

```python
print(...)
```

👉 只是把这些值打印出来，让你观察：

* w 当前是多少
* loss 多大
* gradient 是多少

---

# 🧩 11️⃣ 更新参数（关键动作）

```python
with torch.no_grad():
    w -= learning_rate * w.grad
```

---

## 🔍 先看公式

```text
w = w - 学习率 × 梯度
```

---

## 🔍 第一次代入

```text
w = 0 - 0.1 × (-4)
  = 0 + 0.4
  = 0.4
```

👉 w 变大了

---

## 🧠 为什么这样改？

👉 因为 gradient 是负数：

👉
**说明“往上调 w”会更好**

---

# 🧩 12️⃣ 清梯度

```python
w.grad.zero_()
```

👉 意思：

👉 把梯度清零

---

👉 为什么？

因为 PyTorch 会“累加梯度”

如果不清：

👉 下一轮会乱掉

---

# 🧠 整个循环你可以这样理解

每一轮都在做：

---

## 🪜 四步

```text
1. 用 w 做预测
2. 看错多少（loss）
3. 算怎么改（gradient）
4. 改一点点（update）
```

---

# 🔁 实际过程（你运行时看到的）

```text
w: 0 → 0.4 → 0.72 → 0.97 → ...
loss: 4 → 2.5 → 1.6 → 1.0 → ...
```

---

👉 你要“看到”的是：

👉
**模型在一点点接近正确答案（w → 2）**

---

# 🧠 最后一句话总结

👉
**这段代码在模拟“学习”本质：**

```text
试错 → 看错误 → 调整 → 再试
```

---

# ❗ 如果你还是有点晕（正常）

你可以只记这三句：

👉

* `loss` = 错多少
* `gradient` = 怎么改
* `update` = 真正去改

---

# 🚀 如果你愿意，我可以再帮你做一个升级版

👉 把这个代码改成：

* 有两个参数（更接近神经网络）
* 或者加一个 ReLU（连接你前面的问题）

这样你会彻底串起来。


# ReLU

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt

# 让 matplotlib 支持中文
plt.rcParams["font.family"] = ["SimHei", "Microsoft YaHei"]
plt.rcParams["axes.unicode_minus"] = False  # 解决负号显示问题

# =========================================
# 1. 构造数据：y = x^2
# =========================================
x = torch.linspace(-2, 2, 100).unsqueeze(1)
y = x ** 2

# =========================================
# 2. 模型 A（没有 ReLU）
# =========================================
model_linear = nn.Sequential(
    nn.Linear(1, 10),
    nn.Linear(10, 1)
)

# =========================================
# 3. 模型 B（有 ReLU）
# =========================================
model_relu = nn.Sequential(
    nn.Linear(1, 10),
    nn.ReLU(),
    nn.Linear(10, 1)
)

# =========================================
# 4. 训练函数
# =========================================
def train(model):
    optimizer = optim.SGD(model.parameters(), lr=0.05)
    loss_fn = nn.MSELoss()

    for _ in range(2000):
        pred = model(x)
        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return model(x).detach()


# =========================================
# 5. 训练
# =========================================
y_pred_linear = train(model_linear)
y_pred_relu = train(model_relu)

# =========================================
# 6. 画图
# =========================================
plt.scatter(x.numpy(), y.numpy(), label="真实 y=x²", s=10)
plt.plot(x.numpy(), y_pred_linear.numpy(), label="没有 ReLU", linewidth=2)
plt.plot(x.numpy(), y_pred_relu.numpy(), label="有 ReLU", linewidth=2)

plt.legend()
plt.title("ReLU 的作用对比")
plt.show()